# Extra: six-state estimation

Optional asynchronous-sensor and onboard-estimate comparisons, using a separate recording format. The numbered notebooks cover E1-E7.


In [ ]:
from pathlib import Path
import json
import hashlib
import os
import sys
from dataclasses import replace

# Shared helpers are beside day2; data stays beside these notebooks.
notebook_dir = Path.cwd()
if (notebook_dir / "code/day2").is_dir():
    notebook_dir = notebook_dir / "code/day2"
sys.path.insert(0, str(notebook_dir.resolve().parent))

import numpy as np
from IPython.display import Image, display
from day2.notebook_tools import (
    capture, capture_vertical, completed_capture, read_imu, plot_imu_calibration,
    simulated_imu_pair, display_table, next_attempt, drone_uri,
)

run_id = "trial-01"  # Label this session; each comparison gets a fresh attempt folder.
output = notebook_dir / "data" / run_id
output.mkdir(parents=True, exist_ok=True)
print("Results:", output.resolve())

from kalman.filter import FilterConfig, PositionVelocityKalmanFilter
from race_common.dynamics import position_velocity_model
from day2.extra_notebook_tools import load_capture_overview, plot_capture_overview


## Optional: a six-state drone model

We follow R. Frühwirth, [Application of Kalman filtering to track and vertex fitting (1987)](https://doi.org/10.1016/0168-9002(87)90887-4). The state is $x_k=(\mathbf{p}_k^T,\mathbf{v}_k^T)^T$: three positions and three velocities. With acceleration held constant over $\Delta t$,
$$
\mathbf{v}_{k+1}=\mathbf{v}_k+\Delta t\,\mathbf{a}_k,\qquad
\mathbf{p}_{k+1}=\mathbf{p}_k+\Delta t\,\mathbf{v}_k+\tfrac12\Delta t^2\,\mathbf{a}_k.
$$
The example initializes a state and uncertainty, predicts using this model, then updates with a position measurement. Process input is the acceleration; the position observation supplies the measurement update.

State covariance is $C$, measurement covariance is $V$, and residual covariance is $R$. Diagonal entries of C are state-estimation error variances: m² for position and (m/s)² for velocity. Superscript $k-1$ denotes a prediction using measurements through the preceding step:
$$
x_k^{k-1}=F_{k-1}x_{k-1}+B_{k-1}\mathbf{a}_{k-1},\qquad
C_k^{k-1}=F_{k-1}C_{k-1}F_{k-1}^T+Q_{k-1}.
$$
For position, $H=[I,0]$. The predicted residual, its covariance and the gain are
$$
r_k^{k-1}=m_k-H_kx_k^{k-1},\qquad
R_k^{k-1}=V_k+H_kC_k^{k-1}H_k^T,\qquad
K_k=C_k^{k-1}H_k^T(R_k^{k-1})^{-1}.
$$
The update is $x_k=x_k^{k-1}+K_kr_k^{k-1}$. The code uses the Joseph update for C, described in the optional numerical reference below. Its diagnostics expose the predicted residual as `last_predicted_residual`, its covariance as `last_predicted_residual_covariance`, and V as `last_V`. The chi-square increment is $(r_k^{k-1})^T(R_k^{k-1})^{-1}r_k^{k-1}$. The filtered residual $r_k=m_k-H_kx_k$ is evaluated after the update.

**Optional exploration:** reproduce one prediction/update, identify the six states and distinguish input from measurement. Change either `dt`, initial uncertainty or measurement uncertainty, and explain the effect on the estimate and gain. Keep your prediction and result in the notebook.


In [ ]:
dt = 0.02  # seconds
F, B = position_velocity_model(dt)
display(F, B)
np.set_printoptions(precision=3, suppress=True)

# Standard deviations are the square roots of the covariance diagonal.
def to_errors(C):
    return np.sqrt(np.diag(C))

# filter config holds some constants for the kalman filter, in particular the values
# used to fill the covariance matrices
config = FilterConfig(initial_position_std=0.25, position_noise_std=0.04)
# this objects holds the state-vector and the math to perform the filter
estimator = PositionVelocityKalmanFilter(config)
# Initialize position (0, 0, 1) and velocity (0, 0, 0).
# Initial covariance C_0 is set by config above.
estimator.x[2] = 1.0
print(f"Initial state: {estimator.x} +/- {to_errors(estimator.C)}")
# assume an external acceleration
acceleration = [0.2, 0.0, 0.0]
# predict the next state, 
estimator.predict(dt, acceleration)
print(f"Predicted state: {estimator.x} +/- {to_errors(estimator.C)}")
# update the state with an external position measurement.
# Measurement covariance V is set by position_noise_std above.
# Edit position_noise_std in config above to compare measurement uncertainties.
measurement= np.array([0.02, -0.01, 1.0])
estimator.update_position(measurement)
print(f"Measurement: {measurement} +/- {to_errors(estimator.last_V)}")
print(f"Predicted residual: {estimator.last_predicted_residual} +/- {to_errors(estimator.last_predicted_residual_covariance)}")
print(f"Chi-square increment: {estimator.last_chi2:.3f}")
print(f"Updated state: {estimator.x} +/- {to_errors(estimator.C)}")

## Optional reference: covariance and measurement models

The exercise steps above are sufficient for E1-E7. This section and the software mapping below provide details for inspecting the implementation.

The displayed F and B matrices are derivatives of the predicted state with respect to the previous state and acceleration. The code uses $Q=B\Sigma_aB^T$, with $\Sigma_a=\sigma_a^2I$: an unknown acceleration perturbation held over one interval. In particle tracking, process uncertainty can instead represent effects such as multiple scattering in detector material.

The Joseph update,
$$C_k=(I-K_kH_k)C_k^{k-1}(I-K_kH_k)^T+K_kV_kK_k^T,$$
preserves covariance structure more reliably under floating-point roundoff. The filtered residual has covariance $R_k=V_k-H_kC_kH_k^T$. The model assumes interval-held acceleration and additive, unbiased noise.

The flow model selects X/Y velocity; the simple range model measures world Z with suitable surface and sensor geometry. The onboard state also includes attitude, and its [flow and Lighthouse measurement models](https://www.bitcraze.io/documentation/repository/crazyflie-firmware/master/functional-areas/sensor-to-control/state_estimators/#measurement-models) are more detailed.


### What the live data represent

`cflib` reads the drone's log channels. The course filter runs on the laptop, using onboard position `stateEstimate.x/y/z` as its observation and world-frame acceleration as its input. The helper uses the onboard orientation to rotate body-frame specific force into world coordinates, converts g to m/s² and subtracts gravity.

Onboard position and attitude already use inertial data, so these inputs share sensor information. Live comparisons show consistency; simulation supplies independent truth for measuring error and coverage. The optional reference at the end maps all symbols to code and log channels.


## Optional: inspect a six-state recording

Set `inspection_directory` to a complete saved six-state capture to inspect body-frame specific force, onboard attitude, world-frame acceleration, position and velocity. Those onboard estimates are a separate comparison. E3-E7 use the vertical sensor format described above.


In [ ]:
inspection_directory = None  # Or select the folder containing a checked live capture.
overview = None
capture_dir = inspection_directory

if capture_dir is not None:
    overview = load_capture_overview(capture_dir)
    overview_path = next_attempt(output / "inspection") / "sensor-overview.png"
    plot_capture_overview(overview, overview_path)
    display(Image(filename=str(overview_path)))
    print("Saved:", overview_path)
else:
    print("No six-state recording selected; optional inspection skipped.")


### Reference: from Frühwirth's equations to Crazyflie data

`cflib` connects to the drone and delivers named firmware log values. The course's `PositionVelocityKalmanFilter` performs the six-state calculation on the laptop. These are two separate software layers.

| Paper quantity | Course Python code | Connection to the drone |
| --- | --- | --- |
| $k$, $\Delta t$ | `ReplayFilter.step(time_s, acceleration, position)` | Firmware log timestamps are unwrapped and converted from milliseconds to seconds; consecutive accepted sample times set `dt`. |
| $x_k=(p_x,p_y,p_z,v_x,v_y,v_z)^T$ | `estimator.x` after `update_position`; `.position` and `.velocity` select its two halves | This is the laptop estimate. Its velocity is inferred by the model and updates. |
| $m_k$ for a position update | `position`, passed to `update_position(position)` | The live helper reads `stateEstimate.x`, `.y`, `.z`, in world-frame metres. |
| $\mathbf{a}_{k-1}$, process input | `acceleration`, passed to `predict(dt, acceleration)` | `acc.x`, `.y`, `.z` supply body-frame specific force in g. The helper subtracts the chosen bias, rotates with `stateEstimate.qx`, `.qy`, `.qz`, `.qw`, multiplies by 9.80665, then subtracts world gravity. |
| $F_{k-1}$, $B_{k-1}$ | `position_velocity_model(dt)` | Model matrices calculated on the laptop. |
| $Q_{k-1}$ | `B @ (I * config.acceleration_noise_std**2) @ B.T` | Chosen uncertainty of an unknown acceleration perturbation held over the interval. |
| $H_k$ | `H` in `update_position`: `[I, 0]` | Selects the three position components for comparison with the logged position. |
| $V_k$ | `I * config.position_noise_std**2`; saved as `last_V` | Assumed position-measurement error covariance assigned to the observation used by the course filter. |
| $C_k^{k-1}$, then $C_k$ | `estimator.C` after prediction, then after update | Covariance of the six-component state-estimation error, before and after the observation. |
| $r_k^{k-1}$, $R_k^{k-1}$ | `last_predicted_residual`, `last_predicted_residual_covariance` | Observation minus the predicted measurement; R is the covariance of this predicted residual, saved before the update. |
| $K_k$, $\Delta\chi_k^2$ | `kalman_gain`, `last_chi2`; `config.chi2_gate` sets a rejection threshold | The gain uses predicted state-error covariance C and measurement-error covariance V. Chi-square uses the predicted residual and its covariance R. |

The live path is `cflib` logging → `SampleAssembler` in [live.py](live.py) → `ReplayFilter.step` in [recording.py](../kalman/recording.py) → `predict` and `update_position` in [filter.py](../kalman/filter.py). A `LogConfig` selects channel names; its `data_received_cb` receives `(timestamp, data, config)`, with readings such as `data['stateEstimate.x']`. The helper handles timing, units and the verified firmware motor stop before passing samples to the filter.

The state-estimation error covariance $C$ describes uncertainty in the estimated position and velocity. Each diagonal entry is a **state-estimation error variance**: position variances have units m² and velocity variances (m/s)². For example, $C_{11}=0.01$ m² corresponds to a position-error standard deviation of 0.1 m. Position-velocity covariance entries have units m²/s; off-diagonal entries describe correlated errors. Prediction propagates C with F and adds Q; the measurement update produces the filtered state-estimation error covariance. This is maintained by the course filter, separately from the onboard estimator's covariance. Measurement-error covariance V describes the observations, and predicted-residual covariance R combines their uncertainty with the predicted state uncertainty.

A unit quaternion is a four-component representation of orientation; the logged order is x,y,z,w. The helper uses it to rotate a body-axis vector into the world frame, whose Z axis points upward. At rest, rotation of the accelerometer's specific force gives approximately +g in world Z; subtracting gravity then gives approximately zero translational acceleration.

**Interpretation:** `stateEstimate.*` values are already estimates produced onboard, using positioning and inertial data. Here they become observations $m_k$ of a second, educational filter. The onboard position, quaternion and accelerometer channels can therefore be correlated. Agreement between the two estimates alone cannot establish accuracy or validate the independent-noise assumptions. The prepared simulation supplies known truth for that comparison.

The live helper supplies position observations and acceleration inputs. Flow velocity, range and zero-velocity updates are additional model interfaces used in the other experiments. To connect a new sensor, specify its measurement vector $m_k$, prediction $h_k(x)$, Jacobian $H_k$, covariance $V_k$, frame, units and timestamp.
